# EMF Models Training Pipeline v3

Matches `generate_emf_dataset.py` v4 column schema.

**Features:**
- XGBoost + LightGBM gradient boosting (GPU / CUDA)
- PyTorch 4-layer MLP (CUDA)
- Stacked ensemble with Ridge meta-learner
- Optuna hyperparameter optimization (subsampled for speed)
- Trains for both B-field (µT) and E-field (V/m)
- All artifacts saved to `outputs/`

## 1. Imports & Setup

In [1]:
%pip install -q optuna xgboost lightgbm torch scikit-learn matplotlib pandas numpy


In [2]:
# ── Show Drive root so you know where to upload the CSVs ──────────────
import os

drive_root = '/content/drive/MyDrive'
print("Top-level items in your Google Drive:")
for item in sorted(os.listdir(drive_root))[:30]:
    print(" ", item)
print()
print("ACTION REQUIRED:")
print("  Upload 'grid_emf_dataset.csv' to your Google Drive (or a sub-folder)")
print("  then update DRIVE_CSV in the next cell to its full /content/drive/MyDrive/... path.")


Top-level items in your Google Drive:
   business 
and tech competition and events in oman and uae in a table in next 6month .gsheet
  .ipynb_checkpoints
  10 Tech conference in middle in 3 next month and add to table  (1).gsheet
  10 Tech conference in middle in 3 next month and add to table  (2).gsheet
  10 Tech conference in middle in 3 next month and add to table .gsheet
  1003.pdf
  10848.pdf
  1096.pdf
  1477205523262.pdf
  17030063998665934499905423979960.jpg
  2021-04-24-16-38-49.mp4
  2896.pdf
  3305.pdf
  417_0 (1).pdf
  427.zip
  436.zip
  53343.jpg
  53344.jpg
  693.pdf
  7F6A6A2N.txt
  92699974.txt
  Also when I ask you  hast du kinder answer nein.gdoc
  An Analysis of the Bima Insurance Platform in Oman (1).gdoc
  An Analysis of the Bima Insurance Platform in Oman (10).gdoc
  An Analysis of the Bima Insurance Platform in Oman (11).gdoc
  An Analysis of the Bima Insurance Platform in Oman (12).gdoc
  An Analysis of the Bima Insurance Platform in Oman (13).gdoc
  An Analysi

In [3]:
# ── Copy CSV from Drive to Colab runtime ──────────────────────────────
import shutil, os

DRIVE_CSV = '/content/drive/MyDrive/projects/data/grid_emf_dataset.csv'
CSV_PATH  = 'grid_emf_dataset.csv'

if os.path.exists(DRIVE_CSV):
    shutil.copy(DRIVE_CSV, CSV_PATH)
    print(f"Copied {DRIVE_CSV}  →  {CSV_PATH}")
else:
    raise FileNotFoundError(
        f"Not found: {DRIVE_CSV}\n"
        "Edit DRIVE_CSV above to point to the correct location in your Drive."
    )


Copied /content/drive/MyDrive/projects/data/grid_emf_dataset.csv  →  grid_emf_dataset.csv


In [4]:
import numpy as np
import pandas as pd
import pickle, json, os, warnings, time
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge

import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
import optuna

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("All imports successful.")

All imports successful.


In [3]:
!nvidia-smi

Sat Mar  7 21:47:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Device Configuration

In [5]:
USE_GPU = True   # set False to force CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if USE_GPU and device.type != 'cuda':
    print("WARNING: CUDA requested but not available – falling back to CPU.")
    print("  Install torch+cu126 in a Python 3.12 venv for GPU acceleration.")

print(f"Device : {device}")
if device.type == 'cuda':
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")

XGB_DEVICE = 'cuda' if (USE_GPU and torch.cuda.is_available()) else 'cpu'
LGB_DEVICE = 'gpu'  if (USE_GPU and torch.cuda.is_available()) else 'cpu'

os.makedirs('outputs', exist_ok=True)

Device : cuda
  GPU  : Tesla T4


## 3. Column Schema

In [6]:
TARGET_COLS = ['B_field_uT', 'E_field_V_m']

# Columns to EXCLUDE from features (targets + physics reference + derived)
EXCLUDE_COLS = {
    'B_field_uT', 'E_field_V_m', 'H_field_A_m',
    'B_field_clean_uT', 'E_field_clean_V_m',
    'corona_onset_kV_cm', 'surface_gradient_kV_cm', 'corona_ratio',
    'ICNIRP_E_exceeded', 'ICNIRP_B_exceeded',
}

# Categorical columns to label-encode
CAT_COLS = [
    'configuration', 'feeder', 'substation',
    'weather', 'time_of_day', 'season',
    'profile_type', 'circuit_type',
]

print("Column schema configured.")

Column schema configured.


## 4. Data Loading & Feature Engineering

In [7]:
def load_and_prepare(csv_path: str):
    print(f"Loading {csv_path} ...")
    # Use float32 for numeric columns at read time to halve memory vs float64
    df = pd.read_csv(csv_path)
    # Downcast floats and ints immediately
    float_cols = df.select_dtypes(include='float64').columns
    int_cols   = df.select_dtypes(include='int64').columns
    df[float_cols] = df[float_cols].astype(np.float32)
    df[int_cols]   = df[int_cols].astype(np.int32)
    print(f"  {len(df):,} rows × {len(df.columns)} cols")

    # Additional engineered features (on top of those in the CSV)
    if 'voltage_current_product' not in df.columns:
        df['voltage_current_product'] = df['voltage_kV'] * df['current_A']
    if 'height_to_spacing_ratio' not in df.columns:
        df['height_to_spacing_ratio'] = df['height_m'] / df['phase_spacing_m']
    if 'distance_to_height_ratio' not in df.columns:
        df['distance_to_height_ratio'] = df['distance_m'] / df['height_m']
    if 'sag_to_span_ratio' not in df.columns:
        df['sag_to_span_ratio'] = df['sag_m'] / df['span_length_m']
    if 'log_distance' not in df.columns:
        df['log_distance'] = np.log1p(df['distance_m'].values).astype(np.float32)
    if 'sqrt_distance' not in df.columns:
        df['sqrt_distance'] = np.sqrt(df['distance_m'].values).astype(np.float32)
    if 'inv_distance' not in df.columns:
        df['inv_distance'] = (1 / (df['distance_m'] + 1)).astype(np.float32)
    if 'inv_distance_sq' not in df.columns:
        df['inv_distance_sq'] = (1 / (df['distance_m']**2 + 1)).astype(np.float32)
    if 'temp_humidity_interaction' not in df.columns:
        df['temp_humidity_interaction'] = df['temperature_C'] * df['humidity_pct']
    if 'power_density' not in df.columns:
        df['power_density'] = (df['active_power_MW'] / (df['distance_m'] * df['height_m'] + 1)).astype(np.float32)

    targets = {
        'B': df['B_field_uT'].values.astype(np.float32),
        'E': df['E_field_V_m'].values.astype(np.float32),
    }

    label_encoders = {}
    for col in CAT_COLS:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            label_encoders[col] = le

    feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
    for c in list(feature_cols):
        if df[c].dtype == object:
            feature_cols.remove(c)

    X = df[feature_cols].values.astype(np.float32)
    del df  # free the DataFrame immediately

    print(f"  Features: {X.shape[1]}   Targets: B [{targets['B'].min():.4f}–{targets['B'].max():.2f}]  "
          f"E [{targets['E'].min():.4f}–{targets['E'].max():.2f}]")

    return X, targets, feature_cols, label_encoders

print("load_and_prepare() defined.")


load_and_prepare() defined.


## 5. HPO Objectives (XGBoost & LightGBM)

In [8]:
def objective_xgb(trial, Xtr, ytr, Xv, yv):
    p = {
        'max_depth':        trial.suggest_int('max_depth', 4, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'tree_method': 'hist', 'device': XGB_DEVICE, 'random_state': 42,
    }
    m = xgb.XGBRegressor(**p)
    m.fit(Xtr, ytr, eval_set=[(Xv, yv)], verbose=False)
    return np.sqrt(mean_squared_error(yv, m.predict(Xv)))


def objective_lgb(trial, Xtr, ytr, Xv, yv):
    p = {
        'num_leaves':        trial.suggest_int('num_leaves', 20, 200),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'device_type': LGB_DEVICE, 'random_state': 42, 'verbose': -1,
    }
    m = lgb.LGBMRegressor(**p)
    m.fit(Xtr, ytr, eval_set=[(Xv, yv)])
    return np.sqrt(mean_squared_error(yv, m.predict(Xv)))

print("XGBoost & LightGBM objectives defined.")

XGBoost & LightGBM objectives defined.


## 6. Neural Network Architecture & Training

In [9]:
class EMFNet(nn.Module):
    def __init__(self, in_dim, hidden, dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.GELU(), nn.BatchNorm1d(h), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_nn(model, Xtr, ytr, Xv, yv, epochs, bs, lr):
    crit  = nn.HuberLoss(delta=1.0)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2)

    Xtr_t = torch.from_numpy(Xtr).to(device)
    ytr_t = torch.from_numpy(ytr).to(device)
    Xv_t  = torch.from_numpy(Xv).to(device)
    yv_t  = torch.from_numpy(yv).to(device)

    ds = torch.utils.data.TensorDataset(Xtr_t, ytr_t)
    dl = torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=True,
                                     pin_memory=False, drop_last=True)

    best_val = float('inf')
    patience = 0
    for ep in range(epochs):
        model.train()
        for bx, by in dl:
            opt.zero_grad(set_to_none=True)
            loss = crit(model(bx), by)
            loss.backward()
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            vl = crit(model(Xv_t), yv_t).item()
        if vl < best_val:
            best_val = vl
            patience = 0
        else:
            patience += 1
            if patience >= 12:
                break
    return best_val


def objective_nn(trial, Xtr, ytr, Xv, yv, in_dim):
    nl = trial.suggest_int('n_layers', 2, 5)
    hd = trial.suggest_categorical('hidden_dim', [128, 256, 512, 768])
    dp = trial.suggest_float('dropout', 0.0, 0.35)
    lr = trial.suggest_float('lr', 5e-5, 5e-3, log=True)
    bs = trial.suggest_categorical('batch_size', [4096, 8192, 16384])
    model = EMFNet(in_dim, [hd] * nl, dp).to(device)
    return train_nn(model, Xtr, ytr, Xv, yv, epochs=60, bs=bs, lr=lr)

print("EMFNet, train_nn(), and objective_nn() defined.")

EMFNet, train_nn(), and objective_nn() defined.


## 7. Main Training Pipeline

In [10]:
def nn_predict_batched(model, X, batch_size=32768):
    """Run NN inference in chunks to avoid GPU OOM on large arrays."""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            chunk = torch.from_numpy(X[i:i+batch_size]).to(device)
            preds.append(model(chunk).cpu().numpy())
    return np.concatenate(preds)


def train_all(csv_path: str = 'grid_emf_dataset.csv',
              n_xgb_trials: int = 20,
              n_lgb_trials: int = 20,
              n_nn_trials:  int = 12,
              hpo_sample:   int = 150_000):

    wall0 = time.time()
    print("=" * 80)
    print("EMF MODEL TRAINING PIPELINE v3  (CUDA-accelerated)")
    print("=" * 80)

    X, targets, feat_cols, label_encoders = load_and_prepare(csv_path)

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X).astype(np.float32)

    results = {}

    for tgt in ['B', 'E']:
        print(f"\n{'='*80}\n  Training for {tgt}-field\n{'='*80}")
        y = targets[tgt]
        Xtr, Xte, ytr, yte = train_test_split(X_scaled, y, test_size=0.15, random_state=42)

        # HPO subsample
        ss  = min(hpo_sample, len(Xtr))
        idx = np.random.choice(len(Xtr), ss, replace=False)
        Xh  = Xtr[idx]; yh = ytr[idx]
        Xht, Xhv, yht, yhv = train_test_split(Xh, yh, test_size=0.2, random_state=42)

        print(f"  HPO: {len(Xht):,} train / {len(Xhv):,} val   |   Full: {len(Xtr):,} / {len(Xte):,}")

        # 1. XGBoost ---------------------------------------------------------
        print(f"\n  [1/5] XGBoost HPO ({n_xgb_trials} trials) ...")
        st = optuna.create_study(direction='minimize')
        st.optimize(lambda t: objective_xgb(t, Xht, yht, Xhv, yhv),
                    n_trials=n_xgb_trials, show_progress_bar=True)
        bp_xgb = st.best_params
        print(f"        best RMSE={st.best_value:.5f}")
        xgb_m = xgb.XGBRegressor(**bp_xgb, tree_method='hist',
                                  device=XGB_DEVICE, random_state=42)
        xgb_m.fit(Xtr, ytr)
        xgb_p = xgb_m.predict(Xte)

        # 2. LightGBM --------------------------------------------------------
        print(f"\n  [2/5] LightGBM HPO ({n_lgb_trials} trials) ...")
        st2 = optuna.create_study(direction='minimize')
        st2.optimize(lambda t: objective_lgb(t, Xht, yht, Xhv, yhv),
                     n_trials=n_lgb_trials, show_progress_bar=True)
        bp_lgb = st2.best_params
        print(f"        best RMSE={st2.best_value:.5f}")
        lgb_m = lgb.LGBMRegressor(**bp_lgb, device_type=LGB_DEVICE,
                                  verbose=-1, random_state=42)
        lgb_m.fit(Xtr, ytr)
        lgb_p = lgb_m.predict(Xte)

        # 3. Neural network --------------------------------------------------
        print(f"\n  [3/5] NN HPO ({n_nn_trials} trials) ...")
        st3 = optuna.create_study(direction='minimize')
        st3.optimize(lambda t: objective_nn(t, Xht, yht, Xhv, yhv, Xtr.shape[1]),
                     n_trials=n_nn_trials, show_progress_bar=True)
        bp_nn = st3.best_params
        print(f"        best val_loss={st3.best_value:.6f}")
        h_dims = [bp_nn['hidden_dim']] * bp_nn['n_layers']
        nn_m   = EMFNet(Xtr.shape[1], h_dims, bp_nn['dropout']).to(device)
        train_nn(nn_m, Xtr, ytr, Xte, yte, epochs=120,
                 bs=bp_nn['batch_size'], lr=bp_nn['lr'])
        nn_p = nn_predict_batched(nn_m, Xte)

        # 4. Ensemble --------------------------------------------------------
        print(f"\n  [4/5] Ensemble meta-learner ...")
        # Free GPU memory before large full-dataset inference
        torch.cuda.empty_cache()
        # Force XGBoost/LightGBM to CPU for full-dataset inference to save GPU RAM
        xgb_m.set_params(device='cpu')
        nn_tr_p  = nn_predict_batched(nn_m, Xtr)
        xgb_tr_p = xgb_m.predict(Xtr)
        lgb_tr_p = lgb_m.predict(Xtr)
        Xm_tr = np.column_stack([xgb_tr_p, lgb_tr_p, nn_tr_p])
        Xm_te = np.column_stack([xgb_p, lgb_p, nn_p])
        meta  = Ridge(alpha=1.0)
        meta.fit(Xm_tr, ytr)
        ens_p = meta.predict(Xm_te)

        # 5. Evaluate --------------------------------------------------------
        print(f"\n  [5/5] Evaluation:")
        res = {}
        for name, pred in [('XGBoost', xgb_p), ('LightGBM', lgb_p),
                            ('NeuralNet', nn_p), ('Ensemble', ens_p)]:
            rmse = np.sqrt(mean_squared_error(yte, pred))
            mae  = mean_absolute_error(yte, pred)
            r2   = r2_score(yte, pred)
            mape = np.mean(np.abs((yte - pred) / (np.abs(yte) + 1e-8))) * 100
            print(f"    {name:12s}  RMSE={rmse:9.5f}  MAE={mae:9.5f}  R²={r2:.6f}  MAPE={mape:.2f}%")
            res[name] = dict(rmse=float(rmse), mae=float(mae), r2=float(r2), mape=float(mape))
        results[tgt] = res

        # Save models
        pickle.dump(xgb_m, open(f'outputs/xgb_{tgt.lower()}.pkl', 'wb'))
        pickle.dump(lgb_m, open(f'outputs/lgb_{tgt.lower()}.pkl', 'wb'))
        torch.save(nn_m.state_dict(), f'outputs/nn_{tgt.lower()}.pt')
        pickle.dump(meta,  open(f'outputs/meta_{tgt.lower()}.pkl', 'wb'))
        json.dump({'xgb': bp_xgb, 'lgb': bp_lgb, 'nn': bp_nn},
                  open(f'outputs/best_params_{tgt.lower()}.json', 'w'), indent=2)

        # Plot
        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        for ax, (name, pred) in zip(axes, [('XGBoost', xgb_p), ('LightGBM', lgb_p),
                                            ('NeuralNet', nn_p), ('Ensemble', ens_p)]):
            ax.scatter(yte, pred, alpha=0.15, s=1)
            lo, hi = yte.min(), yte.max()
            ax.plot([lo, hi], [lo, hi], 'r--', lw=2)
            ax.set_xlabel(f'Actual {tgt}-field'); ax.set_ylabel('Predicted')
            ax.set_title(f'{name}  R²={res[name]["r2"]:.4f}')
        fig.tight_layout()
        fig.savefig(f'outputs/{tgt.lower()}_predictions.png', dpi=150)
        plt.show()
        plt.close(fig)

    # Global artifacts
    pickle.dump(scaler, open('outputs/scaler.pkl', 'wb'))
    pickle.dump(label_encoders, open('outputs/label_encoders.pkl', 'wb'))
    json.dump(feat_cols, open('outputs/feature_columns.json', 'w'))
    json.dump({'input_dim': Xtr.shape[1], 'hidden_dims': h_dims,
               'dropout': bp_nn['dropout']},
              open('outputs/nn_architecture.json', 'w'), indent=2)
    json.dump(results, open('outputs/training_results.json', 'w'), indent=2)

    wall = time.time() - wall0
    print(f"\n{'='*80}")
    print(f"  TRAINING COMPLETE  ({wall/60:.1f} min)")
    print(f"  All artifacts → outputs/")
    print(f"{'='*80}")
    return results

print("train_all() defined.")


train_all() defined.


In [12]:
## 8. Run Training

results = train_all(
    csv_path='grid_emf_dataset.csv',
    n_xgb_trials=20,
    n_lgb_trials=20,
    n_nn_trials=12,
    hpo_sample=150_000,
)


EMF MODEL TRAINING PIPELINE v3  (CUDA-accelerated)
Loading grid_emf_dataset.csv ...
  4,329,600 rows × 54 cols
  Features: 44   Targets: B [0.0004–18.30]  E [0.0200–406.00]

  Training for B-field


[I 2026-03-08 00:55:48,460] A new study created in memory with name: no-name-d9090060-c941-45a4-8567-3249bbe27620


  HPO: 120,000 train / 30,000 val   |   Full: 3,680,160 / 649,440

  [1/5] XGBoost HPO (20 trials) ...


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-08 00:55:49,669] Trial 0 finished with value: 0.10806200295996438 and parameters: {'max_depth': 8, 'learning_rate': 0.14821284734921683, 'n_estimators': 499, 'min_child_weight': 1, 'subsample': 0.9224757665944185, 'colsample_bytree': 0.9579995072411112, 'gamma': 3.2262683811480235, 'reg_alpha': 0.004505880170778962, 'reg_lambda': 0.09306717938974018}. Best is trial 0 with value: 0.10806200295996438.
[I 2026-03-08 00:55:50,651] Trial 1 finished with value: 0.13882348007219872 and parameters: {'max_depth': 5, 'learning_rate': 0.0760059224642664, 'n_estimators': 382, 'min_child_weight': 7, 'subsample': 0.6121862787994647, 'colsample_bytree': 0.6141978567035936, 'gamma': 2.509527111584819, 'reg_alpha': 0.001936477264225376, 'reg_lambda': 0.004876380086380841}. Best is trial 0 with value: 0.10806200295996438.
[I 2026-03-08 00:55:52,062] Trial 2 finished with value: 0.17458998425234057 and parameters: {'max_depth': 6, 'learning_rate': 0.010523848743988755, 'n_estimators': 334, 'mi

...

[I 2026-03-08 01:42:52,017] Trial 0 finished with value: 2.630717517367403 and parameters: {'num_leaves': 86, 'learning_rate': 0.05123264707173181, 'n_estimators': 205, 'min_child_samples': 35, 'subsample': 0.8795791567497615, 'colsample_bytree': 0.6350020286962577, 'reg_alpha': 0.020966145608806462, 'reg_lambda': 0.11589136426941664}. Best is trial 0 with value: 2.630717517367403.
[I 2026-03-08 01:43:02,991] Trial 1 finished with value: 2.6872746676319075 and parameters: {'num_leaves': 118, 'learning_rate': 0.014548505517249547, 'n_estimators': 385, 'min_child_samples': 79, 'subsample': 0.9106560548817276, 'colsample_bytree': 0.6205851181092352, 'reg_alpha': 0.008731252732919777, 'reg_lambda': 5.396612328102027}. Best is trial 0 with value: 2.630717517367403.
[I 2026-03-08 01:43:10,069] Trial 2 finished with value: 2.6193364193337882 and parameters: {'num_leaves': 72, 'learning_rate': 0.03466466017128007, 'n_estimators': 392, 'min_child_samples': 38, 'subsample': 0.7495265619347654, '

[I 2026-03-08 01:47:19,604] A new study created in memory with name: no-name-56701a40-cbc1-41c5-b7a9-bd4e6dd0b310



  [3/5] NN HPO (12 trials) ...


  0%|          | 0/12 [00:00<?, ?it/s]

[I 2026-03-08 01:48:44,671] Trial 0 finished with value: 12.917327880859375 and parameters: {'n_layers': 4, 'hidden_dim': 128, 'dropout': 0.16941181853131268, 'lr': 0.0008762412703419851, 'batch_size': 16384}. Best is trial 0 with value: 12.917327880859375.
[I 2026-03-08 01:50:06,816] Trial 1 finished with value: 1.0397119522094727 and parameters: {'n_layers': 4, 'hidden_dim': 128, 'dropout': 0.19376541663938027, 'lr': 0.0004419649669148086, 'batch_size': 4096}. Best is trial 1 with value: 1.0397119522094727.
[I 2026-03-08 01:51:34,678] Trial 2 finished with value: 27.568416595458984 and parameters: {'n_layers': 4, 'hidden_dim': 512, 'dropout': 0.3010225044247404, 'lr': 0.00011236154685415844, 'batch_size': 16384}. Best is trial 1 with value: 1.0397119522094727.
[I 2026-03-08 01:52:57,128] Trial 3 finished with value: 0.9329626560211182 and parameters: {'n_layers': 2, 'hidden_dim': 512, 'dropout': 0.18973529252102084, 'lr': 0.0009776147159508187, 'batch_size': 8192}. Best is trial 3 wi

In [13]:
import json, os

with open('outputs/training_results.json') as f:
    res = json.load(f)

print("=" * 65)
print(f"{'Model':<14} {'RMSE':>10} {'MAE':>10} {'R²':>10} {'MAPE%':>8}")
print("=" * 65)
for tgt, models in res.items():
    print(f"\n  {tgt}-field:")
    for name, m in models.items():
        print(f"  {name:<12}  {m['rmse']:10.5f}  {m['mae']:10.5f}  {m['r2']:10.6f}  {m['mape']:8.2f}")

print("\nSaved artifacts:")
for f in sorted(os.listdir('outputs')):
    print(f"  outputs/{f}")


Model                RMSE        MAE         R²    MAPE%

  B-field:
  XGBoost          0.08696     0.03559    0.997565     10.15
  LightGBM         0.08767     0.03727    0.997525     17.52
  NeuralNet        0.09616     0.05948    0.997022    717.46
  Ensemble         0.08769     0.03648    0.997524     34.58

  E-field:
  XGBoost          2.58112     1.03452    0.997597     17.69
  LightGBM         2.57023     1.04370    0.997618     22.58
  NeuralNet        2.63417     1.09377    0.997498     93.24
  Ensemble         2.55719     1.01985    0.997642     20.28

Saved artifacts:
  outputs/b_predictions.png
  outputs/best_params_b.json
  outputs/best_params_e.json
  outputs/e_predictions.png
  outputs/feature_columns.json
  outputs/label_encoders.pkl
  outputs/lgb_b.pkl
  outputs/lgb_e.pkl
  outputs/meta_b.pkl
  outputs/meta_e.pkl
  outputs/nn_architecture.json
  outputs/nn_b.pt
  outputs/nn_e.pt
  outputs/scaler.pkl
  outputs/training_results.json
  outputs/xgb_b.pkl
  outputs/xgb_e.p